<a href="https://colab.research.google.com/github/nadayali710-spec/Programming-For-AI-Project/blob/main/Hockey_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hockey Team Performance Analysis and Prediction

# Section 1: Web Scraping
**Goal:** scrape every hockey team-season row from the website into a pandas DataFrame (~580 rows, 9 columns), save the raw data with an untouched backup and metadata, then create a working copy with ~5% artificial missing values.

**Source:** the Scrape This Site "Hockey Teams: Forms, Searching and Pagination" practice pages (`https://www.scrapethissite.com/pages/forms/`).

**Outputs:** `hockey_raw.csv`, `hockey_raw_backup.csv`, `scrape_metadata.txt`, `hockey_working_missing.csv`

## Step 1 - Fetch and inspect the HTML

### Cell 1 - Imports
We use `requests` to download pages, `BeautifulSoup` to parse HTML, and `pandas` to build tables.

In [32]:
import time
import datetime as dt
import requests
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup

print("requests", requests.__version__)
print("pandas", pd.__version__)

requests 2.32.4
pandas 2.2.3


### Cell 2 - Fetch one page
We request the first page and check the HTTP status code (`200` means success). The site paginates with the `page_num` query parameter and we ask for 100 rows per page.

In [33]:
BASE_URL = "https://www.scrapethissite.com/pages/forms/"
HEADERS = {"User-Agent": "Mozilla/5.0 (student project - PFAI hockey analysis)"}

resp = requests.get(BASE_URL, params={"page_num": 1, "per_page": 100}, headers=HEADERS, timeout=30)
print("Status code:", resp.status_code)
print("Final URL  :", resp.url)
print("Length     :", len(resp.text), "characters")

Status code: 200
Final URL  : https://www.scrapethissite.com/pages/forms/?page_num=1&per_page=100
Length     : 135648 characters


### Cell 3 - Look at the raw HTML
Before writing a parser we must *see* the structure of the page.

In [34]:
print(resp.text[:1500])

<!doctype html>
<html lang="en">
  <head>
    <meta charset="utf-8">
    <title>Hockey Teams: Forms, Searching and Pagination | Scrape This Site | A public sandbox for learning web scraping</title>
    <link rel="icon" type="image/png" href="/static/images/scraper-icon.png" />

    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <meta name="description" content="Browse through a database of NHL team stats since 1990. Practice building a scraper that handles common website interface components.">

    <link href="https://maxcdn.bootstrapcdn.com/bootstrap/3.3.5/css/bootstrap.min.css" rel="stylesheet" integrity="sha256-MfvZlkHCEqatNoGiOXveE8FIwMzZg4W85qfrfIFBfYc= sha512-dTfge/zgoMYpP7QbHy4gWMEGsbsdZeCXz7irItjcC3sPUFtf0kuFbDz/ixG7ArTxmDjLXDmezHubeNikyKGVyQ==" crossorigin="anonymous">
    <link href='https://fonts.googleapis.com/css?family=Lato:400,700' rel='stylesheet' type='text/css'>
    <link rel="stylesheet" type="text/css" href="/static/css/styles.css">

   

### Cell 4 - Parse with BeautifulSoup and find the table
`BeautifulSoup` turns the HTML text into a tree we can search. The data lives in a `<table class="table">`.

In [35]:
soup = BeautifulSoup(resp.text, "html.parser")
table = soup.find("table", class_="table")
print("Table found:", table is not None)

header_cells = [th.get_text(strip=True) for th in table.find_all("th")]
print("Column headers:", header_cells)

Table found: True
Column headers: ['Team Name', 'Year', 'Wins', 'Losses', 'OT Losses', 'Win %', 'Goals For (GF)', 'Goals Against (GA)', '+ / -']


### Cell 5 - Find the data rows
Each team-season is a `<tr class="team">` row. We count them and print the first one.

In [36]:
rows = table.find_all("tr", class_="team")
print("Rows on this page:", len(rows))
print(rows[0].prettify()[:900])

Rows on this page: 100
<tr class="team">
 <td class="name">
  Boston Bruins
 </td>
 <td class="year">
  1990
 </td>
 <td class="wins">
  44
 </td>
 <td class="losses">
  24
 </td>
 <td class="ot-losses">
 </td>
 <td class="pct text-success">
  0.55
 </td>
 <td class="gf">
  299
 </td>
 <td class="ga">
  264
 </td>
 <td class="diff text-success">
  35
 </td>
</tr>



### Cell 6 - Inspect the cells inside one row
Each `<td>` has a class name (`name`, `year`, `wins`, ...). Those class names give us reliable column names.

In [37]:
first = rows[0]
for td in first.find_all("td"):
    print(f"{td.get('class')[0]:<10} -> {td.get_text(strip=True)!r}")

name       -> 'Boston Bruins'
year       -> '1990'
wins       -> '44'
losses     -> '24'
ot-losses  -> ''
pct        -> '0.55'
gf         -> '299'
ga         -> '264'
diff       -> '35'


## Step 2 - Parse rows and scrape all pages

### Cell 7 - Row parser
One function turns one `<tr>` into a Python dictionary. Column names are tidied (`OT Losses` becomes `OT_Losses`, `Win %` becomes `Win_Pct`, `+ / -` becomes `Diff`).

In [38]:
CLASS_TO_COL = {
    "name": "Team", "year": "Year", "wins": "Wins", "losses": "Losses",
    "ot-losses": "OT_Losses", "pct": "Win_Pct", "gf": "GF", "ga": "GA", "diff": "Diff",
}

def parse_row(tr):
    """Convert one <tr class='team'> into a dict {column: text}."""
    record = {}
    for td in tr.find_all("td"):
        key = td.get("class")[0]
        record[CLASS_TO_COL[key]] = td.get_text(strip=True)
    return record

parse_row(rows[0])

{'Team': 'Boston Bruins',
 'Year': '1990',
 'Wins': '44',
 'Losses': '24',
 'OT_Losses': '',
 'Win_Pct': '0.55',
 'GF': '299',
 'GA': '264',
 'Diff': '35'}

### Cell 8 - `scrape_page` function
Download one page and return a list of row dictionaries.

In [39]:
def scrape_page(page_num, per_page=100):
    r = requests.get(BASE_URL, params={"page_num": page_num, "per_page": per_page},
                     headers=HEADERS, timeout=30)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
    trs = soup.find_all("tr", class_="team")
    time.sleep(0.5)
    return [parse_row(tr) for tr in trs]

test = scrape_page(1)
print(len(test), "rows from page 1")
test[:2]

100 rows from page 1


[{'Team': 'Boston Bruins',
  'Year': '1990',
  'Wins': '44',
  'Losses': '24',
  'OT_Losses': '',
  'Win_Pct': '0.55',
  'GF': '299',
  'GA': '264',
  'Diff': '35'},
 {'Team': 'Buffalo Sabres',
  'Year': '1990',
  'Wins': '31',
  'Losses': '30',
  'OT_Losses': '',
  'Win_Pct': '0.388',
  'GF': '292',
  'GA': '278',
  'Diff': '14'}]

### Cell 9 - Loop over every page
We keep requesting pages until one returns no rows.

In [40]:
all_rows = []
page = 1
while True:
    page_rows = scrape_page(page)
    if not page_rows:
        break
    all_rows.extend(page_rows)
    print(f"page {page}: {len(page_rows)} rows (total {len(all_rows)})")
    page += 1

print("Total scraped rows:", len(all_rows))

page 1: 100 rows (total 100)
page 2: 100 rows (total 200)
page 3: 100 rows (total 300)
page 4: 100 rows (total 400)
page 5: 100 rows (total 500)
page 6: 82 rows (total 582)
Total scraped rows: 582


### Cell 10 - Convert the list of dictionaries to a DataFrame
`pd.DataFrame(list_of_dicts)` builds a 2-D table: each dictionary is one row and each key is a column.

In [41]:
df_raw = pd.DataFrame(all_rows)
df_raw.head()

,Team,Year,Wins,Losses,OT_Losses,Win_Pct,GF,GA,Diff
0,Boston Bruins,1990,44,24,,0.55,299,264,35
1,Buffalo Sabres,1990,31,30,,0.388,292,278,14
2,Calgary Flames,1990,46,26,,0.575,344,263,81
3,Chicago Blackhawks,1990,49,23,,0.613,284,211,73
4,Detroit Red Wings,1990,34,38,,0.425,273,298,-25


### Cell 11 - Fix data types
Scraping gives every value as text. We convert numeric columns to numbers; empty strings become `NaN`.

In [42]:
num_cols = ["Year", "Wins", "Losses", "OT_Losses", "Win_Pct", "GF", "GA", "Diff"]
for col in num_cols:
    df_raw[col] = pd.to_numeric(df_raw[col].replace("", np.nan), errors="coerce")

df_raw["Year"] = df_raw["Year"].astype(int)
df_raw.dtypes

,0
Team,object
Year,int64
Wins,int64
Losses,int64
OT_Losses,float64
Win_Pct,float64
GF,int64
GA,int64
Diff,int64


### Cell 12  workings proof: the notebook prints the full DataFrame
Expected: about 580 rows and 9 columns.

In [43]:
print("Shape:", df_raw.shape)
display(df_raw.head(10))
display(df_raw.tail(5))

Shape: (582, 9)


,Team,Year,Wins,Losses,OT_Losses,Win_Pct,GF,GA,Diff
0,Boston Bruins,1990,44,24,NaN,0.550,299,264,35
1,Buffalo Sabres,1990,31,30,NaN,0.388,292,278,14
2,Calgary Flames,1990,46,26,NaN,0.575,344,263,81
3,Chicago Blackhawks,1990,49,23,NaN,0.613,284,211,73
4,Detroit Red Wings,1990,34,38,NaN,0.425,273,298,-25
5,Edmonton Oilers,1990,37,37,NaN,0.463,272,272,0
6,Hartford Whalers,1990,31,38,NaN,0.388,238,276,-38
7,Los Angeles Kings,1990,46,24,NaN,0.575,340,254,86
8,Minnesota North Stars,1990,27,39,NaN,0.338,256,266,-10
9,Montreal Canadiens,1990,39,30,NaN,0.487,273,249,24


,Team,Year,Wins,Losses,OT_Losses,Win_Pct,GF,GA,Diff
577,Tampa Bay Lightning,2011,38,36,8.0,0.463,235,281,-46
578,Toronto Maple Leafs,2011,35,37,10.0,0.427,231,264,-33
579,Vancouver Canucks,2011,51,22,9.0,0.622,249,198,51
580,Washington Capitals,2011,42,32,8.0,0.512,222,230,-8
581,Winnipeg Jets,2011,37,35,10.0,0.451,225,246,-21


## Step 3 : Save raw data, backup, metadata

### Cell 13 - Save the raw CSV
`hockey_raw.csv` is the **original scraped data**. We never edit it again

In [44]:
df_raw.to_csv("hockey_raw.csv", index=False)
print("Saved hockey_raw.csv")

Saved hockey_raw.csv


### Cell 14 : Untouched backup
A second copy so we can prove that the raw file was never changed. We also store a fingerprint (hash) of the file.

In [45]:
import shutil, hashlib

shutil.copy("hockey_raw.csv", "hockey_raw_backup.csv")

def file_hash(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

raw_hash = file_hash("hockey_raw.csv")
print("MD5 of hockey_raw.csv:", raw_hash)

MD5 of hockey_raw.csv: 783054382c586a985c971d557c7669f4


### Cell 15 - Metadata file
Records where, when and how the data was collected so the work can be reproduced.

In [46]:
meta = f"""Source URL      : {BASE_URL}
Scraped at      : {dt.datetime.now():%Y-%m-%d %H:%M:%S}
Pages scraped   : {page - 1}
Rows            : {df_raw.shape[0]}
Columns         : {df_raw.shape[1]} -> {list(df_raw.columns)}
Year range      : {df_raw['Year'].min()} - {df_raw['Year'].max()}
Libraries       : requests, beautifulsoup4, pandas
MD5 (raw file)  : {raw_hash}
Authors         : Sharjeel & Muazz
"""
with open("scrape_metadata.txt", "w") as f:
    f.write(meta)
print(meta)

Source URL      : https://www.scrapethissite.com/pages/forms/
Scraped at      : 2026-11-28 11:32:11
Pages scraped   : 6
Rows            : 582
Columns         : 9 -> ['Team', 'Year', 'Wins', 'Losses', 'OT_Losses', 'Win_Pct', 'GF', 'GA', 'Diff']
Year range      : 1990 - 2011
Libraries       : requests, beautifulsoup4, pandas
MD5 (raw file)  : 783054382c586a985c971d557c7669f4
Authors         : Sharjeel & Muazz



### Cell 16 - Verify the saved files
Reload the CSV and check it matches the DataFrame, and that the backup is identical.

In [47]:
check = pd.read_csv("hockey_raw.csv")
print("Same shape         :", check.shape == df_raw.shape)
print("Backup == raw      :", file_hash("hockey_raw.csv") == file_hash("hockey_raw_backup.csv"))
print("Missing values (raw):")
print(check.isna().sum())

Same shape         : True
Backup == raw      : True
Missing values (raw):
Team           0
Year           0
Wins           0
Losses         0
OT_Losses    224
Win_Pct        0
GF             0
GA             0
Diff           0
dtype: int64


## Step 4 : Working copy with ~5% missing values
real data is messy, so we **deliberately** punch random holes in a **copy** of the data and later practise imputation.

**Design decisions**
- Only the feature columns Wins, Losses, OT_Losses, GF, GA, Diff receive extra missing values.
- Team, Year and Win_Pct stay complete because `Year` is needed for the chronological split and `Win_Pct` creates our targets.
- A fixed `random_state` makes the result reproducible.

### Cell 17 - Load the raw file into a *working copy*

In [48]:
df_work = pd.read_csv("hockey_raw.csv").copy()
print("Working copy shape:", df_work.shape)
print("Missing before:", int(df_work.isna().sum().sum()))

Working copy shape: (582, 9)
Missing before: 224


### Cell 18 - Choose the columns that may lose values

In [49]:
cols_to_blank = ["Wins", "Losses", "OT_Losses", "GF", "GA", "Diff"]
MISSING_FRAC = 0.05
rng = np.random.default_rng(42)

### Cell 19 : Blank out ~5% of the cells in those columns

In [50]:
for col in cols_to_blank:
    n_missing = int(round(MISSING_FRAC * len(df_work)))
    idx = rng.choice(df_work.index, size=n_missing, replace=False)
    df_work.loc[idx, col] = np.nan

### Cell 20 : Count missing values per column

In [51]:
missing_report = pd.DataFrame({
    "missing_count": df_work.isna().sum(),
    "missing_pct": (df_work.isna().mean() * 100).round(2),
})
missing_report

,missing_count,missing_pct
Team,0,0.00
Year,0,0.00
Wins,29,4.98
Losses,29,4.98
OT_Losses,242,41.58
Win_Pct,0,0.00
GF,29,4.98
GA,29,4.98
Diff,29,4.98


### Cell 21 : Overall percentage of missing cells in the modelled columns

In [52]:
overall = df_work[cols_to_blank].isna().sum().sum() / df_work[cols_to_blank].size * 100
print(f"Missing in the 6 selected columns: {overall:.2f}%")

Missing in the 6 selected columns: 11.08%


### Cell 22 : Save the working copy

In [53]:
df_work.to_csv("hockey_working_missing.csv", index=False)
print("Saved hockey_working_missing.csv", df_work.shape)

Saved hockey_working_missing.csv (582, 9)


### Cell 23 : Confirm the raw file is still untouched
The raw file's hash must still equal the value stored earlier.

In [54]:
print("Raw file unchanged:", file_hash("hockey_raw.csv") == raw_hash)
df_work.head(10)

Raw file unchanged: True


,Team,Year,Wins,Losses,OT_Losses,Win_Pct,GF,GA,Diff
0,Boston Bruins,1990,44.0,24.0,NaN,0.550,299.0,264.0,35.0
1,Buffalo Sabres,1990,31.0,30.0,NaN,0.388,292.0,278.0,14.0
2,Calgary Flames,1990,46.0,26.0,NaN,0.575,344.0,263.0,81.0
3,Chicago Blackhawks,1990,49.0,23.0,NaN,0.613,284.0,211.0,73.0
4,Detroit Red Wings,1990,34.0,38.0,NaN,0.425,273.0,298.0,-25.0
5,Edmonton Oilers,1990,37.0,37.0,NaN,0.463,272.0,272.0,NaN
6,Hartford Whalers,1990,31.0,38.0,NaN,0.388,238.0,276.0,-38.0
7,Los Angeles Kings,1990,46.0,24.0,NaN,0.575,340.0,254.0,86.0
8,Minnesota North Stars,1990,27.0,39.0,NaN,0.338,256.0,266.0,-10.0
9,Montreal Canadiens,1990,39.0,30.0,NaN,0.487,273.0,249.0,24.0
